# Lab 10: Hands-On Lab Using LangChain and LangFlow


In [1]:
!pip install langchain langchain-groq langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [4]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

groq_api_key = "your_api_key"
# Initialize LLM
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7, api_key=groq_api_key)
print("LLM initialized successfully.")


LLM initialized successfully.


## Task 1: Basic Prompt + LLM Chain


In [5]:
# Create a prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}."),
    ("human", "Explain {concept} in simple terms for a beginner.")
])

# Build chain using LCEL (LangChain Expression Language)
chain = prompt | llm | StrOutputParser()

# Invoke the chain
result = chain.invoke({
    "domain": "machine learning",
    "concept": "gradient descent"
})
print(result)

Gradient descent is a fundamental concept in machine learning, and I'm happy to break it down in simple terms.

**What is Gradient Descent?**

Gradient descent is an algorithm used to optimize a model's performance by minimizing the error between its predictions and the actual output. Think of it like a hiker trying to reach the bottom of a valley.

**How Does Gradient Descent Work?**

Imagine you're trying to find the optimal position for a line that best fits a set of data points. Gradient descent works as follows:

1. **Start with an initial guess**: You start with a rough idea of where the optimal line might be.
2. **Calculate the error**: You calculate how far off your current line is from the actual data points. This is called the "loss" or "cost."
3. **Find the direction**: You find the direction of the steepest descent, which is the direction that would decrease the error the most.
4. **Take a step**: You move your line in the direction of the steepest descent, a small step at 

## Task 2: Sequential Chain – Blog Post Generator


In [6]:
from langchain_core.prompts import PromptTemplate

# Step 1: Generate outline
outline_prompt = ChatPromptTemplate.from_template(
    "Create a 5-point outline for a blog post about: {topic}"
)

# Step 2: Expand outline into full post
expand_prompt = ChatPromptTemplate.from_template(
    "Write a short blog post based on this outline:\n{outline}\n\nKeep it under 200 words."
)

# Chain: topic → outline → blog post
outline_chain = outline_prompt | llm | StrOutputParser()
blog_chain = (
    {"outline": outline_chain}
    | expand_prompt
    | llm
    | StrOutputParser()
)

blog_post = blog_chain.invoke({"topic": "How Generative AI is changing healthcare"})
print(blog_post)

**Revolutionizing Healthcare with Generative AI**

Generative AI is transforming the healthcare industry by improving diagnosis, treatment, and patient outcomes. This technology can analyze large datasets to identify patterns and predict patient outcomes, enabling doctors to make more accurate diagnoses and develop effective treatment plans.

Generative AI is also enhancing personalized medicine by creating tailored treatment plans based on individual patient data. AI-powered chatbots and virtual assistants are improving patient engagement and education, while AI-driven pharmacogenomics is increasing medication efficacy and safety.

In addition, Generative AI is streamlining clinical workflows and operations by automating administrative tasks and providing clinical decision support systems for healthcare professionals. AI-assisted data analysis and reporting are also improving patient outcomes.

While there are challenges and limitations to implementing Generative AI in healthcare, its

## Task 3: Conversation with Memory


In [7]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# In-memory store for sessions
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Prompt with history placeholder
prompt_with_history = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor for CS students."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}")
])

chain_with_history = RunnableWithMessageHistory(
    prompt_with_history | llm | StrOutputParser(),
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

config = {"configurable": {"session_id": "lab10_session"}}

# Turn 1
r1 = chain_with_history.invoke({"input": "What is a transformer model?"}, config=config)
print("Turn 1:", r1)
print()

# Turn 2 (references previous context)
r2 = chain_with_history.invoke({"input": "What are the key components you just mentioned?"}, config=config)
print("Turn 2:", r2)

Turn 1: The Transformer model is a type of neural network architecture designed specifically for natural language processing (NLP) tasks. It was introduced in a 2017 research paper by Vaswani et al. and has since become one of the most widely used and successful models in NLP.

**Key Components**

The Transformer model consists of several key components:

1. **Self-Attention Mechanism**: The Transformer model uses a self-attention mechanism to process input sequences. This mechanism allows the model to attend to different parts of the input sequence simultaneously and weigh their importance.
2. **Encoder-Decoder Architecture**: The Transformer model uses an encoder-decoder architecture, where the encoder processes the input sequence and produces a continuous representation, and the decoder generates the output sequence.
3. **Layer Normalization**: The Transformer model uses layer normalization to normalize the output of each layer, which helps to stabilize the training process.
4. **Po

## Task 4: Simple Document Q&A Chain


In [8]:
# Simulate a document (normally you'd load a PDF or website)
document_text = """
LangChain is an open-source framework for building applications powered by large language models (LLMs).
It was created by Harrison Chase in 2022. LangChain provides abstractions for chaining together LLM calls,
managing prompts, storing memory/context, loading documents, and connecting to external tools.
The framework supports multiple LLM providers including OpenAI, Anthropic, Google, and Meta's LLaMA.
Key components include: Chains (sequences of calls), Agents (LLM-driven decision makers),
Tools (external functions the agent can call), and Memory (maintaining conversation history).
LangChain is widely used for building chatbots, document Q&A systems, and AI agents.
"""

qa_prompt = ChatPromptTemplate.from_template(
    """Answer the question based ONLY on the context below.
If the answer is not in the context, say 'Not found in document'.

Context: {context}

Question: {question}

Answer:"""
)

qa_chain = qa_prompt | llm | StrOutputParser()

questions = [
    "Who created LangChain?",
    "What are the key components of LangChain?",
    "When was LangChain released?"
]

for q in questions:
    answer = qa_chain.invoke({"context": document_text, "question": q})
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 50)

Q: Who created LangChain?
A: Harrison Chase.
--------------------------------------------------
Q: What are the key components of LangChain?
A: The key components of LangChain include: 

1. Chains (sequences of calls)
2. Agents (LLM-driven decision makers)
3. Tools (external functions the agent can call)
4. Memory (maintaining conversation history)
--------------------------------------------------
Q: When was LangChain released?
A: 2022
--------------------------------------------------


## Part 2: LangFlow – Visual No-Code LangChain Builder


In [9]:
# Install LangFlow locally and launch it
# Run this cell to install, then run the next cell to start the server

!pip install langflow --quiet
print("LangFlow installed. Run 'langflow run' in your terminal, then open http://localhost:7860")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 18.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 17.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.7/56.7 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.4/93.4 kB 10.7 MB/s eta 

In [10]:
# This launches LangFlow in the background (Jupyter will keep running)
# After running this cell, open http://localhost:7860 in your browser

import subprocess, threading

def run_langflow():
    subprocess.run(["langflow", "run", "--port", "7860"])

thread = threading.Thread(target=run_langflow, daemon=True)
thread.start()

print("LangFlow server starting...")
print("Open your browser at: http://localhost:7860")

LangFlow server starting...
Open your browser at: http://localhost:7860


Exception in thread Thread-3 (run_langflow):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_22835/1844957199.py", line 7, in run_langflow
  File "/usr/lib/python3.12/subprocess.py", line 548, in run
